In [ ]:
# import torch and other basic packages
import torch
import os
import copy
import pandas as pd
import numpy as np
print("PyTorch has version {}".format(torch.__version__))

PyTorch has version 2.10.0+cu128


In [ ]:
# install torch geometric
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.7 MB/s eta 0:00:00


In [ ]:
# here we load the weighted adjacency matrices representing our graphs from a file.
from google.colab import drive
drive.mount('/content/gdrive')
matrices = np.load('gdrive/My Drive/corr_matrices25(US).npy')
print(matrices)

Mounted at /content/gdrive
[[[ 0.92717421  0.11909694  0.26278346 ... -0.21319435 -0.42189892
    0.43251445]
  [-0.2739799   0.76006408  0.22090352 ... -0.05943593  0.1739265
   -0.27555786]
  [-0.39764306  0.13666089  0.94029653 ... -0.52822428 -0.12138742
    0.06818715]
  ...
  [-0.03192546  0.00431826 -0.63387075 ...  0.8957248   0.19976772
   -0.3184808 ]
  [-0.33895998 -0.04948768  0.4810385  ... -0.06151678  0.9131419
    0.03726435]
  [-0.0948312  -0.37285274  0.38180084 ...  0.23385654  0.11459573
    0.89496085]]

 [[ 0.94032004  0.1981559   0.29281831 ... -0.23894834 -0.36565779
    0.33948716]
  [-0.32229208  0.70977936  0.22686424 ...  0.00797273  0.06002967
   -0.33197512]
  [-0.27829669  0.17883014  0.89952321 ... -0.38927154 -0.05196947
    0.18436691]
  ...
  [-0.22182258  0.09498795 -0.50639958 ...  0.88923121  0.19669555
   -0.40830891]
  [-0.33606161 -0.01443988  0.5053314  ... -0.07786212  0.88296465
    0.09344564]
  [-0.14890579 -0.22594286  0.4048447  ...  0.13

In [ ]:
# the purpose of this cell is to create a torch.geometric.data object to contain
# our weighted adjacency matrices. This allows us to apply things like batching later on

# packages for basic math and data manipulation
from scipy.sparse import coo_matrix
import numpy as np
import math
# import the appropriate packages for manipulating data in Torch geometric.
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.utils import dense_to_sparse

# set device to GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'


train_data = []
for i in range(matrices.shape[0]):
  curr_matrix = matrices[i]
  # the entries of the weighted adjacency matrices represent CCM correlations
  # because of the nondeterministic way that this is calculated, the self correlations of a time series is not alway
  # exactly 1. We rectify this
  for j in range(curr_matrix.shape[0]):
    curr_matrix[j][j] = 1
  # Convert curr_matrix to float32 before converting to tensor
  curr_matrix = torch.from_numpy(curr_matrix).to(torch.float32)
  edge_index, edge_attr = dense_to_sparse(curr_matrix)
  # Ensure edge_attr is float32
  edge_attr = edge_attr.to(torch.float32)
  print(edge_index)
  print(edge_attr)
  # some neural network architectures have edge attributes as learnable. We want to keep the weights of the graphs as constant features
  edge_attr.requires_grad_(False)
  # store the current graph as a data object, with constant features for each node.
  data = Data(x=torch.ones((curr_matrix.shape[0], 1), dtype=torch.float32), edge_index=edge_index, edge_attr=edge_attr).to(device)
  train_data.append(data)

Streaming output truncated to the last 5000 lines.
        -4.0288e-01, -1.8235e-01,  4.5019e-01,  3.4542e-01, -8.8890e-02,
         8.6217e-02, -3.8249e-01, -2.7052e-02,  1.2912e-01, -5.5188e-02,
        -7.6592e-02, -6.0553e-01, -5.9301e-01, -4.2848e-01,  2.6951e-01,
        -1.7571e-01, -3.0191e-01, -4.5733e-01,  2.0309e-02,  4.2304e-02,
         2.0552e-01, -2.9720e-01, -5.4119e-02,  1.1832e-01,  2.9668e-01,
         2.6377e-02,  1.0000e+00, -6.8677e-02, -2.3832e-01,  7.3016e-02,
        -7.9927e-02, -2.4270e-01, -3.2444e-01,  3.5767e-02, -1.8785e-02,
        -3.3014e-01, -8.6114e-02, -3.1770e-01,  4.1892e-02, -2.9303e-02,
        -4.7848e-02,  8.1137e-02, -1.5787e-02, -2.7855e-02, -3.0321e-01,
        -1.5401e-01,  4.0008e-01, -1.3899e-01,  6.8258e-02, -3.8921e-01,
        -2.7121e-01, -8.9333e-02, -2.5412e-01,  1.0000e+00, -2.7119e-01,
        -1.4620e-01, -3.5167e-01, -3.6428e-02,  2.8060e-01, -4.0395e-01,
         1.6067e-01, -5.0348e-01, -1.1689e-03, -1.8465e-01, -1.5263e-01,


:In this cell we implement a slight variant of the OCGIN architecture from https://arxiv.org/abs/2012.12931 Section 3.3.2. The only difference is that we replace the GIN convolution layer with a GINE convolution layer from https://arxiv.org/abs/1905.12265 to handle edge attributes. This architecture consists of multiple layers where each layer consists of a multilayer perception with 2 fully connected layers, followed by a GINE convolution and batch normalization.

In [ ]:
# import further torch packages
import torch.nn as nn
import torch.nn.functional as F
# import torch geometric layers and pooling.
from torch_geometric.nn import GINEConv
from torch_geometric.nn import global_add_pool
from torch_geometric.nn import global_max_pool
class GINE(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, num_conv_layers, dropout, edge_attr_dim=1):
        super(GINE, self).__init__()

        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.edge_attr_dim = edge_attr_dim # Store edge_attr_dim

        # Helper to build MLP for GINEConv.
        def build_mlp_for_gine(in_channels, out_channels):
            return nn.Sequential(
                nn.Linear(in_channels, out_channels, bias=False),
                nn.ReLU(),
                nn.Linear(out_channels, out_channels, bias=False)
            )

        # Input to MLP for first layer: node_feature_dim (input_dim)
        self.convs.append(GINEConv(build_mlp_for_gine(input_dim, hidden_dim), edge_dim=self.edge_attr_dim))
        self.bns.append(torch.nn.BatchNorm1d(hidden_dim))

        # Subsequent GINE layers and their batch normalizations
        for i in range(1, num_conv_layers):
          # Input to MLP for subsequent layers: hidden_dim
          self.convs.append(GINEConv(build_mlp_for_gine(hidden_dim, hidden_dim), edge_dim=self.edge_attr_dim))
          self.bns.append(torch.nn.BatchNorm1d(hidden_dim))

        self.hidden_channels = hidden_dim # Store hidden_dim for external use

    def reset_parameters(self):
        for conv in self.convs:
            # reset parameters for MLP layers
            for layer in conv.nn:
                if hasattr(layer, 'reset_parameters'):
                    layer.reset_parameters()
        for bn in self.bns:
            bn.reset_parameters()

    # forward function for neural network
    def forward(self, x, edge_index, batch, edge_attr):
        # Ensure edge_attr has the correct shape for GINEConv: [num_edges, edge_attr_dim]
        # If edge_attr is 1D (e.g., [num_edges]), unsqueeze to [num_edges, 1]
        if edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(-1)

        for i in range(len(self.convs)):
           # Pass edge_attr to GINEConv.
           x = self.convs[i](x, edge_index, edge_attr=edge_attr)
           x = self.bns[i](x)
           x = F.relu(x)
           x = F.dropout(x, p=self.dropout, training=self.training)

        return x

In [ ]:
# function to train the model.
def train(model, data, optimizer, loss_fn):
    # train the model
    model.train()
    loss = 0

    # zero-grad the optimizer
    optimizer.zero_grad()

    # feed data into the model to get node embeddings
    output = model(data.x, data.edge_index, data.batch, data.edge_attr)

    # The loss_fn expects a data object as the first argument and model output as the second.
    loss = loss_fn(data, output)
    loss.backward()
    optimizer.step()

    return loss.item()

In this cell, we implement a custom loss function. The loss function combines the MSE loss associated with individual nodes and the result of an (optional)pooled node embedding

In [ ]:
from torch_geometric.nn import global_mean_pool
from torch_geometric.nn import global_max_pool
def get_combined_loss(weighting, pooling= None):
  def combined_loss_max(data, output):
      pooled_output = global_max_pool(output, data.batch)
      # Calculate graph_target by pooling data.y
      graph_target = global_max_pool(data.y, data.batch)
      graph_loss = torch.nn.MSELoss()(pooled_output, graph_target)
      node_loss = torch.nn.MSELoss()(output, data.y[data.batch])
      return weighting * node_loss + graph_loss
  def combined_loss_mean(data, output):
      pooled_output = global_mean_pool(output, data.batch)
      # Calculate graph_target by pooling data.y
      graph_target = global_mean_pool(data.y, data.batch)
      graph_loss = torch.nn.MSELoss()(pooled_output, graph_target)
      node_loss = torch.nn.MSELoss()(output, data.y[data.batch])
      return weighting * node_loss + graph_loss
  def node_loss_only (data, output):
      return torch.nn.MSELoss()(output, data.y[data.batch])
  # if no pooling is specified return node loss only
  if pooling == None:
      return node_loss_only
  elif pooling == 'max':
    return combined_loss_max
  elif pooling == 'mean':
    return combined_loss_mean
  else:
    print("Invalid pooling method")
    return None


In [ ]:
# In this cell, we will implement a function that trains a model, using a specified loss function, optimizer and other parameters
import torch_geometric.loader
# import the dataloader object
from torch_geometric.loader import dataloader as dataL

def Train_Model(model, optimizer, loss_fn, num_epochs, batch_size, early_stopping):
  # create a dataloader object for batching
  dataloader = dataL.DataLoader(train_data, batch_size=batch_size, shuffle=True)

  # Initialize best loss variables/best model
  best_avg_loss = float('inf')
  best_avg_loss_index = 0
  best_model = None

  for epoch in range(1, num_epochs + 1):
    print(f"Starting Epoch: {epoch:02d}")

    # average loss
    avg_loss = 0
    # number of batches
    batch_num = 0
    # train for the number of batches
    for batch in dataloader:
      loss= train(model, batch, optimizer, loss_fn)
      avg_loss += loss
      batch_num += 1

    # compute the average loss
    avg_loss = avg_loss/batch_num

    # if the average loss is less than the best average loss, record the current model as the best
    if avg_loss < best_avg_loss:
      best_avg_loss = avg_loss
      best_avg_loss_index = epoch
      best_model = copy.deepcopy(model)



    # implement early stopping
    if epoch - best_avg_loss_index > early_stopping:
      print("early stopping")
      break
    # print loss for each epoch
    print(f'Epoch: {epoch:02d}, ' f'Loss: {avg_loss:.6f}', str(batch_num))
  return best_model, best_avg_loss, best_avg_loss_index


In the below cell, we implement a function that will give an anomaly score for each graph. Then it will return the indices of the graphs with the anomaly scores over a certain threshhold (percentile).

The anomaly scores of each graph are based on the value of a specified loss function

In [ ]:
def FindOutliers(model, threshold, loss_fn_metric):
  # Set model to evaluation mode
  model.eval()
  # records anomaly scores for each model
  model_vals = []
  # disable gradient computation
  with torch.no_grad():
    for i in range(len(train_data)):
       # Move data to the same device as the model
      data = train_data[i].to(device)
      # Create a batch tensor (all zeros for a single graph)
      batch_tensor = torch.zeros(data.x.shape[0], dtype=torch.long, device=data.x.device)
      # compute the node embeddings
      node_embeddings_output = model(data.x, data.edge_index, batch_tensor, data.edge_attr)
      # compute the loss using the provided loss_fn_metric
      loss = loss_fn_metric(data, node_embeddings_output)
      model_vals.append(loss.item())
  model_scores = np.asarray(model_vals)
  # compute mean and standard deviation of model scores and print
  mean = np.mean(model_scores)
  std = np.std(model_scores)
  print("mean:" + str(mean) + "std:" + str(std))

  # Compute indices of data objects with anomaly score above threshholds
  outlier_thresh = np.percentile(model_scores, threshold)
  ext_indices = np.where(model_scores > outlier_thresh)

  model_preds = np.zeros((len(train_data)))
  model_preds[ext_indices] = 1
  return model_preds



In following cell, we initialize the model used for the OCGIN (E) architecture. We implement some of the strategies from the original paper on deep one-class learning (https://proceedings.mlr.press/v80/ruff18a.html), such as setting the bias to zero and freezing in each fully-connected layer (thereby preventing hypersphere collapse).

We will also set the y value of each data object (target) to the average of the embeddings of each of the graphs upon initialization of the model.

In [ ]:
# Import global_add_pool here to ensure it's available
from torch_geometric.nn import global_add_pool
def InitializeOCGINE(num_layers, dropout, hidden_dim):
  # Determine edge_attr_dim dynamically from the first data object
  # Assuming all train_data items have the same edge_attr dimensionality
  first_data_item = train_data[0]
  # edge_attr is 1D tensor of shape [num_edges] so its feature dimension is 1.
  edge_attr_dim = 1 if first_data_item.edge_attr.dim() == 1 else first_data_item.edge_attr.size(1)
  # initialize the model
  model = GINE(
           1,                   # input_dim (Node feature size)
            hidden_dim,                 # hidden_dim (Hidden layer size)
           num_conv_layers=num_layers,
           dropout=dropout, # Pass dropout as a keyword argument
           edge_attr_dim=edge_attr_dim # Pass edge_attr_dim to GIN constructor

  )
  # put the model on the GPU and reset the parameters
  model.to(device) # Move the model to the GPU

  model.reset_parameters()

  # Iterate through all parameters and set bias terms to zero, then freeze them.
  # This ensures bias terms are effectively removed (set to zero and not updated).
  for name, param in model.named_parameters():
    if 'bias' in name:
        print(f"Setting bias {name} to zero and freezing.")
        param.data.fill_(0)
        param.requires_grad = False

  # Initialize sum for pooled embeddings. Use model.hidden_channels to ensure correct size.
  pooled_embeddings_sum = torch.zeros((model.hidden_channels)).to(device)

  for i in range(len(train_data)):
     # Move graph data to the same device as the model
     data = train_data[i].to(device)

     # GINE model outputs node embeddings first
     node_embeddings = model(data.x, data.edge_index, data.batch, data.edge_attr)
     # Then apply pooling to get graph-level embeddings
     # For a single graph in data, data.batch will be all zeros, so this sums node features.
     graph_embedding = global_add_pool(node_embeddings, data.batch)

     # Squeeze to remove the batch dimension [1, hidden_channels] -> [hidden_channels]
     pooled_embeddings_sum += graph_embedding.squeeze()

  # compute the mean and detach from the computational graph, since it will be our (constant) training objective
  mean = (pooled_embeddings_sum / torch.tensor(len(train_data), dtype=torch.float, device=device)).detach()
  # Assign target 'y' for each node based on the computed mean.
  for i in range(len(train_data)):
    num_nodes = train_data[i].x.shape[0]
    # Repeat the mean vector for each node in the current graph
    train_data[i].y = mean.unsqueeze(0).repeat(num_nodes, 1)

  return model





In this cell, we initialize the neural networks used for graph anomaly detection from https://arxiv.org/abs/2112.10063. We then apply the training below.

First, we initialize a teacher neural network and freeze its parameters. Then we initialize a student network with the exact same architecture. We set the y value (target) of each data object to be the value of the teacher neural network, so that we can train the student network to mimic the teacher network in a cell below.

As mentioned in section 4.1 of the previous paper, theoretically, we can choose any GNN architecture we want for the teacher/student. We choose the same architecture as was used for OCGIN(E) above, due to the fact that we are interested in global properties of neural networks, and the GIN(E) layer is able to capture these due to the connection of GIN with Weisfaler-Lehman test.   

In [ ]:
# this function will initialize the teacher and student model, set the target to the values of the teacher model, and then return the student model
def InitializeGlocalKD(num_layers, dropout, hidden_dim):


  # Determine edge_attr_dim dynamically from the first data object
  # Assuming all train_data items have the same edge_attr dimensionality
  first_data_item = train_data[0]
  # edge_attr is 1D tensor of shape [num_edges] so its feature dimension is 1.
  edge_attr_dim = 1 if first_data_item.edge_attr.dim() == 1 else first_data_item.edge_attr.size(1)
  # initialize the teacher model
  teacher_model = GINE(
           1,                   # input_dim (Node feature size)
            hidden_dim,                 # hidden_dim (Hidden layer size)
           num_conv_layers=num_layers,
           dropout=dropout, # Pass dropout as a keyword argument
           edge_attr_dim=edge_attr_dim # Pass edge_attr_dim to GIN constructor

  )
  # move teacher model to the GPU
  teacher_model.to(device)
  # we freeze the teacher model parameters

  for name, param in teacher_model.named_parameters():


      param.requires_grad = False

  # we initialize our student model
  model = GINE(
           1,                   # input_dim (Node feature size)
            hidden_dim,                 # hidden_dim (Hidden layer size)
           num_conv_layers=num_layers,
           dropout=dropout, # Pass dropout as a keyword argument
           edge_attr_dim=edge_attr_dim # Pass edge_attr_dim to GIN constructor

  )
  # move the student model to the GPU
  model.to(device)
  model.reset_parameters()
  # set the target for the training of the student model to the value of the teacher model
  for i in range(len(train_data)):
    # Get node embeddings from teacher model for a single graph
    node_embeddings_teacher = teacher_model(train_data[i].x, train_data[i].edge_index, None, train_data[i].edge_attr)
    # Create a batch tensor for a single graph (all nodes belong to batch 0)
    batch_single_graph = torch.zeros(node_embeddings_teacher.shape[0], dtype=torch.long, device=device)
    # set the target for the training to the value of the teacher model
    train_data[i].y = node_embeddings_teacher
    train_data[i] = train_data[i].to(device)
  # return the student model
  return model


In [ ]:
# in this cell we train the OCGIN(E) and compute the anomaly scores for each data point.

import os
import itertools
import torch.nn as nn
import torch.nn.functional as F



# hyperparameters
batch_sizes_list = [25, 50, 100]
learning_rates_list = [0.00001, 0.0001, 0.001, 0.01]
num_layers_list = [2, 3]
weight_decays_list = [0.001, 0.0001, 0.00001, 0.000001]
model_index = 0
all_preds = []




for num_layers, weight_decay in itertools.product(num_layers_list, weight_decays_list):

  for bs, lr in itertools.product(batch_sizes_list, learning_rates_list):
      print(f"wd={weight_decay} layers={num_layers} bs={bs} lr={lr}")
      # We use mean pooled loss (weighting=0) as the primary loss for OCGINE here following the architecture from the original paper
      loss_fn = get_combined_loss(0, pooling='mean')
      # initialize the model and create optimizer
      model = InitializeOCGINE(num_layers, 0.3, 10)
      optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
      # train the model with the current hyperparameters
      curr_model, best_loss, best_index = Train_Model(model, optimizer, loss_fn, 200, bs, 40)

      # compute the outliers
      curr_model_preds = FindOutliers(curr_model, 97.5, loss_fn_metric=get_combined_loss(0, pooling='mean') )
      # save the current model
      PATH = os.path.join(os.getcwd(), "model_state_GIN" + str(model_index) + ".pth")
      torch.save(curr_model.state_dict(), PATH)
      model_index+= 1
      all_preds.append(curr_model_preds)
# save the outliers detected for each choice of hyperparameters
np.save("ONESHOTGINPREDSUS(25).npy", np.asarray(all_preds))


Streaming output truncated to the last 5000 lines.
Epoch: 146, Loss: 2.644328 43
Starting Epoch: 147
Epoch: 147, Loss: 2.720455 43
Starting Epoch: 148
Epoch: 148, Loss: 2.777280 43
Starting Epoch: 149
Epoch: 149, Loss: 2.764774 43
Starting Epoch: 150
Epoch: 150, Loss: 2.670887 43
Starting Epoch: 151
Epoch: 151, Loss: 2.692491 43
Starting Epoch: 152
Epoch: 152, Loss: 2.671140 43
Starting Epoch: 153
Epoch: 153, Loss: 2.872174 43
Starting Epoch: 154
Epoch: 154, Loss: 2.760994 43
Starting Epoch: 155
Epoch: 155, Loss: 2.653520 43
Starting Epoch: 156
Epoch: 156, Loss: 2.624596 43
Starting Epoch: 157
Epoch: 157, Loss: 2.767471 43
Starting Epoch: 158
Epoch: 158, Loss: 2.641009 43
Starting Epoch: 159
Epoch: 159, Loss: 2.792030 43
Starting Epoch: 160
Epoch: 160, Loss: 2.584385 43
Starting Epoch: 161
Epoch: 161, Loss: 2.460498 43
Starting Epoch: 162
Epoch: 162, Loss: 2.722423 43
Starting Epoch: 163
Epoch: 163, Loss: 2.729352 43
Starting Epoch: 164
Epoch: 164, Loss: 2.832318 43
Starting Epoch: 165

In [ ]:
# in this cell we train the GlocalKD (GINE) according to the procedures described in the paper  https://arxiv.org/abs/2112.10063. and compute the outliers for each
# choice of hyperparameter.
import os
import itertools
# the hyperparameters
batch_sizes_list = [25, 50, 100]
learning_rates_list = [0.00001, 0.0001, 0.001, 0.01]
num_layers_list = [2, 3]
# this is the parameter lambda which weights the loss for the global embedding/individual node loss
weightings_list = [0.1, 0.5, 0.9]
best_models = []
# save the outliers predicted for each set of hyperparameters
all_preds = []
model_index = 0
for num_layers in num_layers_list:

  for bs, lr, weighting_val in itertools.product(batch_sizes_list, learning_rates_list, weightings_list):
      print("layers=" + str(num_layers) + " bs=" + str(bs) +  " lr="+ str(lr) + " weighting=" + str(weighting_val))
      # initialize the model and optimizer
      model = InitializeGlocalKD(num_layers, 0.3, 10)
      optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
      # the loss in the GlocalKD paper is a linear combination of the max embedding loss and the individual node losses
      loss_fn = get_combined_loss(weighting_val, pooling='max')
      # compute the model trained with the current hyperparameters
      curr_model, best_loss, best_index = Train_Model(model, optimizer, loss_fn, 200, bs, 40)
      # use the loss function as a metric for outliers
      loss_metric_for_outliers = get_combined_loss(weighting_val, pooling='max')
      # compute the outliers
      curr_model_preds = FindOutliers(curr_model, 97.5, loss_metric_for_outliers)
      all_preds.append(curr_model_preds)
      # save the current model
      PATH = os.path.join(os.getcwd(), "KDGIN" + str(model_index) + ".pth")
      torch.save(curr_model.state_dict(), PATH)
      model_index+= 1
  #  save the outliers detected for each choice of hyperparameter
  np.save("KDGINUS(25).npy", np.asarray(all_preds))

Streaming output truncated to the last 5000 lines.
Epoch: 75, Loss: 0.749787 86
Starting Epoch: 76
Epoch: 76, Loss: 0.758057 86
Starting Epoch: 77
Epoch: 77, Loss: 0.753721 86
Starting Epoch: 78
Epoch: 78, Loss: 0.743752 86
Starting Epoch: 79
Epoch: 79, Loss: 0.746156 86
Starting Epoch: 80
Epoch: 80, Loss: 0.748755 86
Starting Epoch: 81
Epoch: 81, Loss: 0.753548 86
Starting Epoch: 82
Epoch: 82, Loss: 0.753182 86
Starting Epoch: 83
Epoch: 83, Loss: 0.742667 86
Starting Epoch: 84
Epoch: 84, Loss: 0.758921 86
Starting Epoch: 85
Epoch: 85, Loss: 0.756738 86
Starting Epoch: 86
Epoch: 86, Loss: 0.741240 86
Starting Epoch: 87
Epoch: 87, Loss: 0.746320 86
Starting Epoch: 88
Epoch: 88, Loss: 0.743314 86
Starting Epoch: 89
Epoch: 89, Loss: 0.750855 86
Starting Epoch: 90
Epoch: 90, Loss: 0.758050 86
Starting Epoch: 91
Epoch: 91, Loss: 0.737276 86
Starting Epoch: 92
Epoch: 92, Loss: 0.740591 86
Starting Epoch: 93
Epoch: 93, Loss: 0.743956 86
Starting Epoch: 94
Epoch: 94, Loss: 0.741989 86
Starting